In [ ]:
%pip install -q ipylab

# Panels and Widgets

In [ ]:
import ipywidgets as ipw

import ipylab

## Panel

A `Panel` widget is the same as a `ipywidget.Box`, but with a `Title` that is used when the panel is added to the application shell.

In [ ]:
panel = ipylab.Panel(children=[ipw.Dropdown()])

To quickly add the panel to the JupyterLab *shell* main area:

In [ ]:
panel.app.shell.expand_right()

In [ ]:
t = panel.add_to_shell(mode=ipylab.InsertMode.split_right)

In [ ]:
ma = t.result()

In [ ]:
# closable isn't synchronised from the panel, but we can set it using set_attribute.
t = ma.set_attribute("title.closable", False)

The title label can be updated as required.

In [ ]:
panel.title.label = "This panel has a dropdown"

We can close the panel and the view will disappear.

In [ ]:
panel.close()

**Note for ipynb:** Do not await the returned task in any cell because it will block the event loop indenfinitely.

In the case of sliders and other widgets that fit on a single line, they can even be added to the top area directly:

In [ ]:
slider = ipw.IntSlider()
panel.app.shell.add(slider, area=ipylab.Area.top)

We can also remove it from the top area when we are done.

In [ ]:
slider.close()

## SplitPanel
A split panel is a subclass of Panel that provides a draggable border between widgets, whose orientatation can be either horizontal or vertical.
Let's create a `SplitPanel` with a few widgets inside.

In [ ]:
split_panel = ipylab.SplitPanel()
progress = ipw.IntProgress(
    value=7,
    min=0,
    max=100,
    step=1,
    description="Loading:",
    bar_style="info",
    orientation="horizontal",
    layout={"height": "30px"},
)
slider_ctrl = ipw.IntSlider(
    min=0,
    max=100,
    step=1,
    description="Slider Control:",
)

# link the slider to the progress bar
ipw.jslink((slider_ctrl, "value"), (progress, "value"))

# add the widgets to the split panel
split_panel.children = [progress, slider_ctrl]
ipw.Box(children=[split_panel], layout={"height": "100px"})

In [ ]:
split_panel.title.label = "A SplitPanel "
split_panel.title.icon_class = "jp-PythonIcon"

> As an alternative to `icon_class`, a `Panel` can also use custom [icons](./icons.ipynb).

In [ ]:
split_panel.add_to_shell(area=ipylab.Area.main, mode=ipylab.InsertMode.split_bottom)

The orientation can be updated on the fly:

In [ ]:
split_panel.orientation = "horizontal"

Let's put it back to `vertical`

In [ ]:
split_panel.orientation = "vertical"

Just like with boxes, we can add an existing widget (the progress bar) more than once:

In [ ]:
split_panel.children += (progress,)

Or add a new widget:

In [ ]:
play = ipw.Play(min=0, max=100, step=1, description="Press play")
ipw.jslink((play, "value"), (slider_ctrl, "value"))
split_panel.children += (play,)

## Left and Right Areas

The same `SplitPanel` widget (or `Panel` or `Widget`) can be added to the left area:

In [ ]:
split_panel.add_to_shell(area=ipylab.Area.left, rank=1000)
split_panel.app.shell.expand_left()

As well as on the right area:

In [ ]:
split_panel.add_to_shell(area=ipylab.Area.right, rank=1000)
split_panel.app.shell.expand_right()

In [ ]:
t = split_panel.app.shell.collapse_right()

## JupyterFrontEnd (app)
`JupyterFrontEnd` is a *singleton* type of object, meaning there is only one instance available in a running kernel. So calling `JupyterFrontEnd()` will provide the same object every time. The instance of this object is given the name `app` is made available as a property on most classes.

In [ ]:
app = ipylab.JupyterFrontEnd()
assert app is panel.app is split_panel.app  # noqa: S101

### [Dialogs](https://jupyterlab.readthedocs.io/en/stable/extension/ui_helpers.html#dialogs)

Various dialogs are provided to interact with the users, possibly getting a returned value. Outside of notebooks, the response can be awaited.

#### Cancelling

Except for the `showDialog` function, cancelling will raise an `IpylabFrontendError`.

In [ ]:
app = ipylab.JupyterFrontEnd()

#### Item

In [ ]:
t = app.dialog.get_item("Select an item", [1, 2, 3])

In [ ]:
t.result()

#### Boolean

In [ ]:
t = app.dialog.get_boolean("Select boolean value")

In [ ]:
t.result()

#### Number

In [ ]:
t = app.dialog.get_number("Provide a numeric value")

In [ ]:
t.result()

#### Text

In [ ]:
t = app.dialog.get_text("Enter text")

In [ ]:
t.result()

#### Password

Note: messaging is not encrypted

In [ ]:
t = app.dialog.get_password("Provide a password")

In [ ]:
t.result()

In [ ]:
t = app.dialog.show_dialog(
    "A custom dialog",
    "It returns the raw result, and there is no cancellation",
    buttons=[
        {
            "ariaLabel": "Accept",
            "label": "Accept",
            "iconClass": "",
            "iconLabel": "",
            "caption": "Accept the result",
            "className": "",
            "accept": "accept",
            "actions": [],
            "displayType": "warn",
        },
        {
            "ariaLabel": "Cancel",
            "label": "Cancel",
            "iconClass": "",
            "iconLabel": "",
            "caption": "",
            "className": "",
            "accept": "cancel",
            "actions": [],
            "displayType": "default",
        },
    ],
    checkbox={  # Optional checkbox in the dialog footer
        "label": "check me",  # Checkbox label
        "caption": "check me I'm magic",  # Checkbox tooltip
        "className": "my-checkbox",  # Additional checkbox CSS class
        "checked": True,  # Default checkbox state
    },
    hasClose=False,
)

In [ ]:
t.result()

**This one has some Ipywidgets in the body!**

In [ ]:
t = app.dialog.show_dialog(body=ipw.VBox([ipw.HTML("<b>SomeTitle</b>"), ipw.FloatSlider()]))

In [ ]:
t = app.dialog.show_error_message(
    "My error",
    "Please acknowledge",
    buttons=[
        {
            "ariaLabel": "Accept",
            "label": "Accept",
            "iconClass": "",
            "iconLabel": "",
            "caption": "Accept the result",
            "className": "",
            "accept": "accept",
            "actions": [],
            "displayType": "warn",
        }
    ],
)

In [ ]:
t.result()

### [File dialogs](https://jupyterlab.readthedocs.io/en/stable/extension/ui_helpers.html#file-dialogs)

In [ ]:
app = ipylab.JupyterFrontEnd()

In [ ]:
t = app.file_dialog.get_open_files()

In [ ]:
t.result()

In [ ]:
t = app.file_dialog.get_existing_directory()

In [ ]:
t.result()